In [1]:
import os
from tqdm.auto import tqdm
from pathlib import Path

import pandas as pd

from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# RUN ONCE: change working directory to project root
cwd = Path.cwd()

pwd = cwd.parent

os.chdir(pwd)
print(f"Changed working directory to {pwd}")

if Path.cwd() != pwd:
    raise RuntimeError(f"Failed to change working directory to {pwd}")

Changed working directory to /Users/jdk/projects/recruiting-reader


In [3]:
# create driver
from src.scraper.driver import make_driver
driver = make_driver(headless=True)

In [4]:
from src.config import PORTAL_2025, PORTAL_2024
from src.scraper.link_extractor import scrape_portal_player_links


# urls_2024 = scrape_portal_player_links(driver, PORTAL_2024)
# print(f"Found {len(urls_2024)} player transfer portal links for 2024 class.")

urls_2025 = scrape_portal_player_links(driver, PORTAL_2025)
print(f"Found {len(urls_2025)} player transfer portal links for 2025 class.")

# all_urls = list(dict.fromkeys(urls_2025 + urls_2024))
# print("total portal urls:", len(all_urls))

Clicking 'Load More':   0%|          | 0/200 [00:00<?, ?click/s]

Found 3010 player transfer portal links for 2025 class.


In [5]:
# urls_2025[2:]
# first two results erroneous cbssports.com links

In [9]:
## tests
# from src.scraper.player_scraper import scrape_player
# p = scrape_player(driver, 'https://247sports.com/player/eric-singleton-jr-46134398/college-297869/')
# p

# from src.utils.tests import test_timeline
# events = test_timeline(driver, "https://247sports.com/player/howard-sampson-46129672/college-310950/")
# len(events)

# from src.config import DEBUG
# from src.scraper.player_scraper import scrape_player

# rows = []
# for u in tqdm(urls_2025[2:102]):
#     try:
#         d = scrape_player(driver, u)
#         rows.append(d)
#     except Exception as e:
#         print("FAIL", u, e)

# portal_df = pd.DataFrame(rows)
# portal_df.isna().sum()

In [ ]:
from src.scraper.player_scraper import scrape_player
from src.storage.cache import load_cache, append_cache, scrape_one
from src.config import CACHE_PATH, WORKERS

# Load cache + filter todo
cache = load_cache(CACHE_PATH)
# todo = [u for u in all_urls if u not in cache]
todo = [u for u in urls_2025[2:] if u not in cache]
print("cached:", len(cache), "todo:", len(todo))

# Start with cached rows
rows = list(cache.values())

# Parallel scrape todo with checkpointing
fails = []
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(scrape_one, u): u for u in todo}
    for fut in tqdm(as_completed(futs), total=len(futs)):
        u = futs[fut]
        try:
            d = fut.result()
            rows.append(d)
            append_cache(d, CACHE_PATH)
        except Exception as e:
            fails.append((u, str(e)))
            print("FAIL:", u, e)

print("done. scraped:", len(rows), "failed:", len(fails))

portal_df = pd.DataFrame(rows)

# Optional: save a clean CSV snapshot too
portal_df.to_csv("portal_scrape_2024_2025.csv", index=False)

portal_df.head()

In [5]:
driver.quit()